### Optuna
- 최적의 파라미터를 찾기 위한 라이브러리
- create_study() 라는 내장 함수를 이용하여 특정 모델의 최적의 파라미터를 서치
- GridSearchCV에 비해서 속도 면에서 우세
    - GridSearchCV는 파라미터의 모든 조합을 fit하고 검증의 결과를 확인
    - Optuna는 확률 기반 -> 모든 조합을 활용하지는 않는다.
- 조합의 수 제어
    - GridSearchCV : params로 제어
    - Optuna : n_trials 매개변수로 제어

In [ ]:
# 라이브러리 설치 
# !pip install optuna

  Obtaining dependency information for optuna from https://files.pythonhosted.org/packages/ab/f3/e5fcd5d9b15771ed6dc10e3a7eeddc672e418f4f4c4653d216cc1d857e2d/optuna-4.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for alembic>=1.5.0 from https://files.pythonhosted.org/packages/d2/29/6533c317b74f707ea28f8d633734dbda2119bbadfc61b2f3640ba835d0f7/alembic-1.18.4-py3-none-any.whl.metadata
  Obtaining dependency information for colorlog from https://files.pythonhosted.org/packages/6d/c1/e419ef3723a074172b68aaa89c9f3de486ed4c2399e2dbd8113a4fdcaf9e/colorlog-6.10.1-py3-none-any.whl.metadata
  Obtaining dependency information for sqlalchemy>=1.4.2 from https://files.pythonhosted.org/packages/c9/18/280d00654cc19d1fccf236fa5070f6dd04b84dde6f1b2e637bde0ff340a7/sqlalchemy-2.0.50-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for tqdm from https://files.pythonhosted.org/packages/eb/75/1a0392bcc21c44dcdf87b3cf2d137e7829be2c083a1e38d44efca3d57a16/tqdm-4.68.2-py3-


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import optuna 
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer

c:\study\multicampus_practice\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# iris 데이터 로드 
X, y = load_iris(return_X_y=True)

In [4]:
# objective 함수를 생성 : 파라미터의 조합, 파이프, 폴드, 검증 방법 
def objective(trial):
    # SVC 모델의 파라미터 조합을 생성 
    # suggest_XXX
        # suggest_int, suggest_float : 정수, 실수 형태의 파라미터 조합 (시작값, 종료값, log매개변수)
            # log 매개변수 : False기본값, True로 변경하면 로그 스케일로 조합을 생성(실수형태에서 사용)
        # suggest_categorical : 특정 범주 조합 
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1.0, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    model = SVC(C=C, gamma = gamma, kernel = kernel)

    pipe = Pipeline(
        [
            ('std', StandardScaler()), 
            ('clf', model)
        ]
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = cross_val_score(pipe, X, y, cv = cv, scoring= make_scorer(f1_score, average='macro'))

    return scores.mean()

In [5]:
study = optuna.create_study(
    direction= 'maximize', 
    study_name='class_ml_tuning'
)
study.optimize(
    objective, n_trials=30, show_progress_bar=True
)

[I 2026-06-14 22:01:44,432] A new study created in memory with name: class_ml_tuning
Best trial: 3. Best value: 0.979849:  27%|██▋       | 8/30 [00:00<00:00, 38.30it/s]

[I 2026-06-14 22:01:44,475] Trial 0 finished with value: 0.8651321398124466 and parameters: {'C': 4.231742026558853, 'gamma': 0.00020328206257516694, 'kernel': 'rbf'}. Best is trial 0 with value: 0.8651321398124466.
[I 2026-06-14 22:01:44,501] Trial 1 finished with value: 0.9599331662489557 and parameters: {'C': 1.4975789615395185, 'gamma': 0.28279298343626674, 'kernel': 'rbf'}. Best is trial 1 with value: 0.9599331662489557.
[I 2026-06-14 22:01:44,524] Trial 2 finished with value: 0.953216374269006 and parameters: {'C': 0.9478476077052331, 'gamma': 0.3715422309009122, 'kernel': 'rbf'}. Best is trial 1 with value: 0.9599331662489557.
[I 2026-06-14 22:01:44,547] Trial 3 finished with value: 0.9798486114275586 and parameters: {'C': 7.98137230187987, 'gamma': 0.02795862333534002, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,572] Trial 4 finished with value: 0.9187055447655798 and parameters: {'C': 0.030492688347739914, 'gamma': 0.000158452707

Best trial: 3. Best value: 0.979849:  47%|████▋     | 14/30 [00:00<00:00, 35.61it/s]

[I 2026-06-14 22:01:44,673] Trial 8 finished with value: 0.8651321398124466 and parameters: {'C': 0.01921090741611845, 'gamma': 0.0004145874519732113, 'kernel': 'rbf'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,708] Trial 9 finished with value: 0.9599331662489556 and parameters: {'C': 0.3277540077692264, 'gamma': 0.09029854043172086, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,741] Trial 10 finished with value: 0.8651321398124466 and parameters: {'C': 0.0012691987537550066, 'gamma': 0.013731402134909268, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,768] Trial 11 finished with value: 0.9732664995822891 and parameters: {'C': 9.32826939190306, 'gamma': 0.04137562197451366, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,793] Trial 12 finished with value: 0.9798486114275586 and parameters: {'C': 7.235447544230732, 'gamma': 0

Best trial: 3. Best value: 0.979849:  73%|███████▎  | 22/30 [00:00<00:00, 37.22it/s]

[I 2026-06-14 22:01:44,875] Trial 15 finished with value: 0.9732664995822891 and parameters: {'C': 8.944645367593358, 'gamma': 0.0027337537235887013, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,900] Trial 16 finished with value: 0.9661728917348785 and parameters: {'C': 2.383427527753824, 'gamma': 0.08678782841207217, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,925] Trial 17 finished with value: 0.9464651527809422 and parameters: {'C': 0.10514782024830531, 'gamma': 0.7862435353025363, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,954] Trial 18 finished with value: 0.9661728917348785 and parameters: {'C': 5.189414984513867, 'gamma': 0.004860944433171666, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:44,978] Trial 19 finished with value: 0.9663805979595453 and parameters: {'C': 1.0094400944604662, 'gamma': 

Best trial: 3. Best value: 0.979849: 100%|██████████| 30/30 [00:00<00:00, 37.04it/s]

[I 2026-06-14 22:01:45,082] Trial 23 finished with value: 0.9661728917348785 and parameters: {'C': 4.372629476618384, 'gamma': 0.013500897114509698, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:45,108] Trial 24 finished with value: 0.9661728917348785 and parameters: {'C': 2.6014420896267727, 'gamma': 0.02106253479346782, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:45,132] Trial 25 finished with value: 0.9661728917348785 and parameters: {'C': 5.594513875924029, 'gamma': 0.15778908644157938, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:45,157] Trial 26 finished with value: 0.9663805979595453 and parameters: {'C': 0.9485792485179212, 'gamma': 0.005639463852609891, 'kernel': 'linear'}. Best is trial 3 with value: 0.9798486114275586.
[I 2026-06-14 22:01:45,185] Trial 27 finished with value: 0.9599331662489557 and parameters: {'C': 0.4809247686883431, 'gamma': 

In [6]:
# 최적의 파라미터 값을 출력 
study.best_params

{'C': 7.98137230187987, 'gamma': 0.02795862333534002, 'kernel': 'linear'}

In [7]:
# 최적의 스코어 확인 
study.best_value

0.9798486114275586